# Pemodelan EduPath AI v2 Berbasis Topic Accuracy

Notebook ini digunakan untuk melatih model simulasi EduPath AI menggunakan dataset dummy berbasis topic_accuracy.

Tujuan notebook ini adalah mengetahui apakah sistem dapat memprediksi kebutuhan remedial pengguna berdasarkan pemahaman per topik.

## 1. Import Library

Pada tahap ini kita memanggil library yang dibutuhkan untuk membaca data, melakukan preprocessing, melatih model, dan mengevaluasi hasil model.

In [1]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load Dataset

Membaca dataset dummy EduPath v2.

Dataset ini berisi data interaksi belajar pengguna seperti topic_accuracy, correct_answers, wrong_answers, attempt_count, study_duration_minutes, dan needs_remedial.

In [3]:
df = pd.read_csv("../dataset/processed/edupath_dataset_v2_dummy_topic_accuracy.csv")

print("Dataset Preview:")
display(df.head())

print("Shape Dataset:")
print(df.shape)

Dataset Preview:


,user_id,subject,topic,difficulty_level,total_questions,correct_answers,wrong_answers,topic_accuracy,attempt_count,study_duration_minutes,pre_test_score,post_test_score,improvement_score,mastery_level,needs_remedial,recommended_action,remediation_material_id,interaction_date
0,U00001,Business,Marketing Basics,easy,15,14,1,93.33,1,34,86.97,88.87,1.90,mastered,0,continue,M0077,2026-03-30
1,U00002,Digital Literacy,File Management,hard,20,16,4,80.00,3,75,74.61,76.51,1.90,mastered,0,continue,M0069,2026-05-23
2,U00003,Programming,Loops,hard,5,1,4,20.00,4,57,15.95,39.98,24.03,weak,1,remedial,M0023,2026-05-30
3,U00004,Science,Matter,medium,10,8,2,80.00,3,53,77.44,78.91,1.47,mastered,0,continue,M0058,2026-04-27
4,U00005,Mathematics,Geometry,hard,10,5,5,50.00,5,73,45.85,70.03,24.18,weak,1,remedial,M0007,2026-01-22


Shape Dataset:
(2000, 18)


## 3. Memahami Struktur Dataset

Pada tahap ini kita melihat struktur dataset, nama kolom, tipe data, dan jumlah data kosong.

Tujuannya adalah memastikan dataset sudah siap digunakan untuk proses Machine Learning.

Kolom seperti subject, topic, dan difficulty_level masih berupa teks sehingga nanti perlu diubah menjadi angka menggunakan encoding.

In [4]:
print("Informasi Dataset:")
df.info()

print("\nDaftar Kolom:")
print(df.columns.tolist())

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  2000 non-null   object 
 1   subject                  2000 non-null   object 
 2   topic                    2000 non-null   object 
 3   difficulty_level         2000 non-null   object 
 4   total_questions          2000 non-null   int64  
 5   correct_answers          2000 non-null   int64  
 6   wrong_answers            2000 non-null   int64  
 7   topic_accuracy           2000 non-null   float64
 8   attempt_count            2000 non-null   int64  
 9   study_duration_minutes   2000 non-null   int64  
 10  pre_test_score           2000 non-null   float64
 11  post_test_score          2000 non-null   float64
 12  improvement_score        2000 non-null   float64
 13  mastery_level            2000 non-null   object 
 14  needs

## 4. Pemeriksaan Missing Value

Sebelum model dilatih, memastikan tidak ada data kosong karena dapat menyebabkan error atau menurunkan kualitas model.

Jika ditemukan missing value

In [5]:
print("Missing Values Sebelum Cleaning:")
print(df.isnull().sum())

df = df.dropna()

print("\nMissing Values Setelah Cleaning:")
print(df.isnull().sum())

print("\nShape Setelah Cleaning:")
print(df.shape)

Missing Values Sebelum Cleaning:
user_id                    0
subject                    0
topic                      0
difficulty_level           0
total_questions            0
correct_answers            0
wrong_answers              0
topic_accuracy             0
attempt_count              0
study_duration_minutes     0
pre_test_score             0
post_test_score            0
improvement_score          0
mastery_level              0
needs_remedial             0
recommended_action         0
remediation_material_id    0
interaction_date           0
dtype: int64

Missing Values Setelah Cleaning:
user_id                    0
subject                    0
topic                      0
difficulty_level           0
total_questions            0
correct_answers            0
wrong_answers              0
topic_accuracy             0
attempt_count              0
study_duration_minutes     0
pre_test_score             0
post_test_score            0
improvement_score          0
mastery_level        

## 5. Menghindari Data Leakage

Data leakage adalah kondisi ketika model menggunakan informasi yang seharusnya belum tersedia pada saat prediksi dilakukan.

Dalam konteks EduPath AI, model seharusnya memprediksi apakah pengguna perlu remedial berdasarkan data awal seperti hasil quiz, akurasi topik, jumlah percobaan, dan durasi belajar.

Oleh karena itu, beberapa kolom tidak digunakan sebagai feature utama karena merupakan informasi setelah proses evaluasi atau setelah remedial dilakukan, seperti:

- post_test_score
- improvement_score
- mastery_level
- recommended_action
- remediation_material_id
- interaction_date

Model hanya akan menggunakan data yang tersedia sebelum sistem memberikan keputusan remedial.

## 6. Menentukan Feature dan Target

Feature adalah indikator yang digunakan model untuk belajar.

Target adalah hasil yang ingin diprediksi oleh model.

Pada project EduPath AI, target yang digunakan adalah needs_remedial.

Arti target:

- 0 = Tidak perlu remedial
- 1 = Perlu remedial

Feature yang digunakan adalah indikator awal pembelajaran seperti subject, topic, difficulty_level, total_questions, correct_answers, wrong_answers, topic_accuracy, attempt_count, study_duration_minutes, dan pre_test_score.

## 6. Menentukan Feature dan Target

Feature adalah indikator yang digunakan model untuk belajar.

Target adalah hasil yang ingin diprediksi oleh model.

Pada project EduPath AI, target yang digunakan adalah needs_remedial.

Arti target:

- 0 = Tidak perlu remedial
- 1 = Perlu remedial

Feature yang digunakan adalah indikator awal pembelajaran seperti subject, topic, difficulty_level, total_questions, correct_answers, wrong_answers, topic_accuracy, attempt_count, study_duration_minutes, dan pre_test_score.

In [6]:
features = [
    "subject",
    "topic",
    "difficulty_level",
    "total_questions",
    "correct_answers",
    "wrong_answers",
    "topic_accuracy",
    "attempt_count",
    "study_duration_minutes",
    "pre_test_score"
]

target = "needs_remedial"

X = df[features].copy()
y = df[target].copy()

print("Feature X:")
display(X.head())

print("\nTarget y:")
display(y.head())

Feature X:


,subject,topic,difficulty_level,total_questions,correct_answers,wrong_answers,topic_accuracy,attempt_count,study_duration_minutes,pre_test_score
0,Business,Marketing Basics,easy,15,14,1,93.33,1,34,86.97
1,Digital Literacy,File Management,hard,20,16,4,80.00,3,75,74.61
2,Programming,Loops,hard,5,1,4,20.00,4,57,15.95
3,Science,Matter,medium,10,8,2,80.00,3,53,77.44
4,Mathematics,Geometry,hard,10,5,5,50.00,5,73,45.85



Target y:


0    0
1    0
2    1
3    0
4    1
Name: needs_remedial, dtype: int64

## 7. Analisis Distribusi Target

Pada tahap ini kita melihat jumlah data pada masing-masing kelas target.

Hal ini penting untuk mengetahui apakah dataset seimbang atau tidak.

Jika jumlah kelas 0 dan kelas 1 terlalu jauh berbeda, maka model dapat menjadi bias terhadap kelas yang jumlahnya lebih banyak.

In [7]:
print("Distribusi needs_remedial:")
print(y.value_counts())

print("\nDistribusi dalam persentase:")
print(y.value_counts(normalize=True) * 100)

Distribusi needs_remedial:
needs_remedial
0    1132
1     868
Name: count, dtype: int64

Distribusi dalam persentase:
needs_remedial
0    56.6
1    43.4
Name: proportion, dtype: float64


## 8. Encoding Data Kategori

Beberapa kolom pada feature masih berbentuk teks, seperti subject, topic, dan difficulty_level.

Model Machine Learning tidak dapat membaca data teks secara langsung, sehingga data kategori perlu diubah menjadi angka.

Pada tahap ini digunakan LabelEncoder untuk mengubah nilai kategori menjadi angka agar dapat diproses oleh model.

In [8]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "subject",
    "topic",
    "difficulty_level"
]

label_encoders = {}

for col in categorical_columns:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])
    label_encoders[col] = encoder

print("Feature Setelah Encoding:")
display(X.head())

Feature Setelah Encoding:


,subject,topic,difficulty_level,total_questions,correct_answers,wrong_answers,topic_accuracy,attempt_count,study_duration_minutes,pre_test_score
0,0,21,0,15,14,1,93.33,1,34,86.97
1,1,11,1,20,16,4,80.00,3,75,74.61
2,4,20,1,5,1,4,20.00,4,57,15.95
3,5,22,2,10,8,2,80.00,3,53,77.44
4,3,16,1,10,5,5,50.00,5,73,45.85


## 9. Membagi Data Training dan Testing

Pada tahap ini dataset dibagi menjadi dua bagian, yaitu data training dan data testing.

Data training digunakan untuk melatih model agar mengenali pola dari data.

Data testing digunakan untuk menguji apakah model mampu memprediksi data baru yang belum pernah dilihat sebelumnya.

Pembagian data yang digunakan adalah:

- 80% data training
- 20% data testing

Parameter stratify digunakan agar distribusi kelas target needs_remedial tetap seimbang pada data training dan testing.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1600, 10)
X_test : (400, 10)
y_train: (1600,)
y_test : (400,)


## 10. Training Model Random Forest

Pada tahap ini model Machine Learning dilatih menggunakan algoritma Random Forest.

Random Forest dipilih karena cocok untuk data tabular dan dapat menangkap hubungan antar fitur yang lebih kompleks.

Model akan belajar dari data training untuk memprediksi apakah pengguna perlu remedial atau tidak berdasarkan indikator seperti topic_accuracy, correct_answers, wrong_answers, attempt_count, study_duration_minutes, dan pre_test_score.

In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Training model selesai.")

Training model selesai.


## 11. Prediksi Data Testing

Setelah model dilatih, langkah berikutnya adalah melakukan prediksi terhadap data testing.

Data testing adalah data yang belum pernah dilihat model saat proses training, sehingga dapat digunakan untuk mengukur kemampuan model dalam memprediksi data baru.

In [11]:
y_pred = model.predict(X_test)

print("Hasil prediksi:")
print(y_pred[:20])

Hasil prediksi:
[0 0 0 1 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 1]


## 12. Evaluasi Model

Pada tahap ini model dievaluasi menggunakan data testing.

Evaluasi dilakukan untuk mengetahui seberapa baik model memprediksi kebutuhan remedial pengguna.

Metrik yang digunakan:

- Accuracy: persentase prediksi yang benar.
- Confusion Matrix: melihat jumlah prediksi benar dan salah.
- Classification Report: menampilkan precision, recall, dan f1-score.

Dalam konteks EduPath AI, recall pada kelas 1 penting karena kelas 1 berarti pengguna perlu remedial.

In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Accuracy:")
print(round(accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(report)

Accuracy:
100.0 %

Confusion Matrix:
[[226   0]
 [  0 174]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       226
           1       1.00      1.00      1.00       174

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



### Interpretasi Hasil Evaluasi

Model Random Forest memperoleh accuracy sebesar 100% pada dataset dummy EduPath v2.

Hasil ini menunjukkan bahwa model mampu mempelajari pola pada dataset simulasi dengan sangat baik. Namun, hasil ini perlu dipahami secara hati-hati karena dataset yang digunakan adalah dataset dummy yang dibuat dengan pola yang terstruktur.

Kemungkinan besar label `needs_remedial` memiliki hubungan yang sangat kuat dengan fitur seperti `topic_accuracy`, `correct_answers`, `wrong_answers`, dan `pre_test_score`.

Oleh karena itu, hasil accuracy 100% tidak boleh diklaim sebagai performa dunia nyata, melainkan sebagai bukti konsep bahwa struktur data berbasis topic accuracy dapat digunakan untuk membangun sistem diagnosis remedial pada EduPath AI.

## 13. Analisis Feature Importance

Feature Importance digunakan untuk mengetahui fitur mana yang paling berpengaruh terhadap keputusan model.

Pada EduPath AI, analisis ini penting karena kita ingin mengetahui apakah `topic_accuracy` benar-benar menjadi indikator utama dalam menentukan kebutuhan remedial pengguna.

In [13]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance)

,Feature,Importance
6,topic_accuracy,0.478523
9,pre_test_score,0.343618
5,wrong_answers,0.106533
4,correct_answers,0.055398
3,total_questions,0.009829
2,difficulty_level,0.003693
8,study_duration_minutes,0.001239
1,topic,0.000515
0,subject,0.000466
7,attempt_count,0.000187


### Interpretasi Feature Importance

Berdasarkan hasil Feature Importance, fitur yang paling berpengaruh terhadap prediksi kebutuhan remedial adalah `topic_accuracy` dengan nilai importance sebesar 47.85%.

Hal ini menunjukkan bahwa persentase pemahaman pengguna pada suatu topik menjadi indikator utama dalam menentukan apakah pengguna perlu remedial atau tidak.

Fitur kedua yang berpengaruh adalah `pre_test_score` sebesar 34.36%. Artinya, nilai awal pengguna sebelum remedial juga memiliki hubungan kuat terhadap kebutuhan remedial.

Selain itu, `wrong_answers` dan `correct_answers` juga berpengaruh terhadap keputusan model, karena kedua fitur tersebut berkaitan langsung dengan hasil evaluasi pengguna.

Hasil ini mendukung konsep utama EduPath AI, yaitu sistem pembelajaran adaptif yang menganalisis pemahaman pengguna berdasarkan performa per topik, lalu memberikan rekomendasi penguatan materi pada bagian yang belum dikuasai.

## 14. Simulasi Prediksi Pengguna Baru

Pada tahap ini dilakukan simulasi prediksi terhadap satu data pengguna baru.

Simulasi ini bertujuan untuk menunjukkan bagaimana model dapat digunakan dalam aplikasi EduPath AI.

Contoh data pengguna baru berisi informasi hasil evaluasi seperti subject, topic, tingkat kesulitan, jumlah soal, jawaban benar, jawaban salah, akurasi topik, jumlah percobaan, durasi belajar, dan nilai awal.

Model kemudian akan memprediksi apakah pengguna tersebut perlu remedial atau tidak.

In [14]:
sample_user = pd.DataFrame([{
    "subject": "Programming",
    "topic": "Loops",
    "difficulty_level": "hard",
    "total_questions": 10,
    "correct_answers": 4,
    "wrong_answers": 6,
    "topic_accuracy": 40,
    "attempt_count": 2,
    "study_duration_minutes": 35,
    "pre_test_score": 45
}])

sample_encoded = sample_user.copy()

for col in categorical_columns:
    sample_encoded[col] = label_encoders[col].transform(sample_encoded[col])

prediction = model.predict(sample_encoded[features])[0]
prediction_proba = model.predict_proba(sample_encoded[features])[0]

print("Input Pengguna Baru:")
display(sample_user)

print("\nHasil Prediksi:")
if prediction == 1:
    print("Pengguna perlu remedial.")
else:
    print("Pengguna tidak perlu remedial.")

print("\nProbabilitas Prediksi:")
print(prediction_proba)

Input Pengguna Baru:


,subject,topic,difficulty_level,total_questions,correct_answers,wrong_answers,topic_accuracy,attempt_count,study_duration_minutes,pre_test_score
0,Programming,Loops,hard,10,4,6,40,2,35,45



Hasil Prediksi:
Pengguna perlu remedial.

Probabilitas Prediksi:
[0. 1.]


### Interpretasi Simulasi Prediksi

Berdasarkan simulasi pengguna baru, model memprediksi bahwa pengguna perlu remedial.

Hal ini sesuai dengan data input, karena pengguna hanya memperoleh `topic_accuracy` sebesar 40%, dengan 4 jawaban benar dari 10 soal dan nilai awal `pre_test_score` sebesar 45.

Hasil ini menunjukkan bahwa model mampu menggunakan indikator pemahaman per topik untuk menentukan kebutuhan remedial.

Probabilitas prediksi `[0. 1.]` menunjukkan bahwa model sangat yakin pengguna termasuk ke dalam kelas `needs_remedial = 1`.

Namun, karena dataset yang digunakan adalah dataset dummy, tingkat keyakinan ini perlu dipahami sebagai hasil simulasi, bukan performa nyata di lingkungan pengguna sebenarnya.

## 15. Menyimpan Model dan Encoder

Setelah model berhasil dilatih dan diuji, langkah berikutnya adalah menyimpan model ke dalam file.

Model disimpan agar dapat digunakan kembali tanpa perlu melakukan training ulang.

Selain model, encoder juga perlu disimpan karena kolom kategori seperti subject, topic, dan difficulty_level sudah diubah menjadi angka menggunakan LabelEncoder.

Jika nanti aplikasi menerima input baru dalam bentuk teks, encoder digunakan untuk mengubah teks tersebut menjadi angka sesuai aturan saat training.

In [16]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "edupath_topic_accuracy_model.pkl"
encoder_path = MODEL_DIR / "edupath_topic_accuracy_encoders.pkl"

joblib.dump(model, model_path)
joblib.dump(label_encoders, encoder_path)

print("Model berhasil disimpan ke:", model_path)
print("Encoder berhasil disimpan ke:", encoder_path)

Model berhasil disimpan ke: d:\Kuliah\Dicoding\Github\EduPath-AI\models\edupath_topic_accuracy_model.pkl
Encoder berhasil disimpan ke: d:\Kuliah\Dicoding\Github\EduPath-AI\models\edupath_topic_accuracy_encoders.pkl


# Kesimpulan

Berdasarkan hasil pemodelan menggunakan dataset dummy EduPath v2 berbasis `topic_accuracy`, model Random Forest berhasil memprediksi kebutuhan remedial pengguna berdasarkan indikator pemahaman per topik.

Fitur yang paling berpengaruh adalah `topic_accuracy`, diikuti oleh `pre_test_score`, `wrong_answers`, dan `correct_answers`.

Hal ini mendukung konsep utama EduPath AI, yaitu sistem tidak hanya melihat nilai akhir, tetapi juga menganalisis pemahaman pengguna pada setiap topik pembelajaran.

Model ini digunakan sebagai simulasi struktur data ideal EduPath AI. Karena dataset yang digunakan masih berupa data dummy, hasil akurasi tidak dapat diklaim sebagai performa nyata di dunia pengguna sebenarnya. Pada pengembangan berikutnya, dataset ini perlu digantikan atau dilengkapi dengan data pengguna nyata dari aplikasi EduPath AI.